Import

In [1]:
import pandas as pd
import numpy as np

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

Chargement des données

In [2]:
df = pd.read_csv("data/raw/train.csv")

Feature Engineering

##  Préparation des données

Dans cette étape, nous préparons les données pour l’évaluation du modèle.

Nous utilisons les mêmes variables que lors de l’entraînement afin de garantir une comparaison cohérente.

L’objectif est de mesurer la performance réelle du modèle sur des données qu’il n’a jamais vues.

In [3]:
df["TotalSF"] = df["GrLivArea"] + df["TotalBsmtSF"]
df["HouseAge"] = df["YrSold"] - df["YearBuilt"]
df["QualityScore"] = df["OverallQual"] * df["OverallCond"]

features = [
    "GrLivArea",
    "OverallQual",
    "GarageCars",
    "TotalBsmtSF",
    "YearBuilt",
    "TotalSF",
    "HouseAge",
    "QualityScore"
]

df_model = df[features + ["SalePrice"]].dropna()

X = df_model[features]
y = df_model["SalePrice"]

Split + modèle

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

model = RandomForestRegressor(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

preds = model.predict(X_test)

Évaluation

##  Résultats du modèle

Nous utilisons trois métriques pour évaluer la performance :

- **MAE** : erreur moyenne en euros
- **RMSE** : pénalise les grandes erreurs
- **R² Score** : mesure la qualité globale du modèle

 Plus le R² est proche de 1, plus le modèle est performant.

Le modèle Random Forest montre une bonne capacité de prédiction.

In [5]:
mae = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))
r2 = r2_score(y_test, preds)

print("📊 ÉVALUATION DU MODÈLE")
print("----------------------")
print("MAE  :", round(mae, 2), "€")
print("RMSE :", round(rmse, 2), "€")
print("R²   :", round(r2, 3))

📊 ÉVALUATION DU MODÈLE
----------------------
MAE  : 18144.91 €
RMSE : 28830.66 €
R²   : 0.892


In [6]:
baseline = np.mean(y_test)
baseline_preds = [baseline] * len(y_test)

baseline_mae = mean_absolute_error(y_test, baseline_preds)

print("Baseline MAE :", baseline_mae)
print("Modèle MAE   :", mae)

Baseline MAE : 61903.204846124965
Modèle MAE   : 18144.905683504563


##  Comparaison avec une baseline

Nous comparons notre modèle à une baseline simple qui prédit toujours la moyenne des prix.

 Si notre modèle est meilleur que cette baseline, cela signifie qu’il apprend réellement des données.

Ici, le modèle Random Forest est largement supérieur à cette approche naïve.

Chargement des données